# Phase 2 (Step 1 & 2): PeptideCLM-2 Embeddings + Joint K-Means Clustering

Takes the Phase 1 output (`data/processed/dataset.parquet`, built by `clamp-data run-all` — see `data/README.md` and `MD_design_docs/09_phase1_implementation_design.md`) and runs the first two Phase 2 steps from `MD_design_docs/08_implementation_roadmap.md`:

1. **Base embeddings** — pass the deduplicated peptide list through the base pretrained PeptideCLM-2 encoder, mean-pooled final-layer output.
2. **Joint K-Means clustering** — sweep k=4–6 on the union of all HC50 and MIC peptides (never split the tasks independently here — that's what causes leakage across the shared encoder later).

The remaining Phase 2 steps (homology cross-check + leakage validation via MMSeqs2 / graph-part) are a separate notebook/script — not covered here.

Designed to run on Colab with a GPU runtime.

## 0. Setup

In [ ]:
# Uncomment on a fresh Colab runtime. torch is already installed on GPU runtimes.
# !pip install -q transformers rdkit scikit-learn pandas pyarrow

## 1. Load the Phase 1 dataset

Upload `data/processed/dataset.parquet` from the CLAMP repo (Colab's file browser, or mount Google Drive) and point `DATASET_PATH` at it.

In [ ]:
DATASET_PATH = "dataset.parquet"  # update after uploading

## 2. Model configuration & initialization

`trust_remote_code=True` is required — PeptideCLM-2 ships custom modeling/tokenizer code and will not load correctly through vanilla `transformers` without it (`MD_design_docs/02_peptideclm_transfer_learning_plan.md` §5.2).

In [ ]:
import torch
import numpy as np
import pandas as pd
from rdkit import Chem
from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

# mlm-large (337M params) as a fixed encoder for Phase 2 embeddings. This doesn't
# need to match whatever variant Phase 3's model-selection pilot eventually picks
# (hybrid-small/mtr-small/mlm-small/mlm-large are all still candidates there) --
# it just needs to be one fixed, reasonable encoder for clustering.
# Switch to 'aaronfeller/peptideclm-2-hybrid-small' for faster iteration.
MODEL_NAME = "aaronfeller/peptideclm-2-mlm-large"

print(f"Loading tokenizer & model from HF Hub: {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Defensive check: custom SMILES tokenizers sometimes don't define a pad token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token else "[PAD]"

model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, use_safetensors=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print(f"Loaded on device: {device}")

## 3. Mean-pooled embedding extraction

`max_length=2048` matches the known config for the sibling PeptideMTR tokenizer (`MD_design_docs/03_task5_warmstart_encoder_ablation_plan.md`) — set explicitly rather than relying on the tokenizer's default truncation behavior.

In [ ]:
def extract_mean_pooled_embeddings(
    smiles_list: list[str],
    batch_size: int = 32,
    max_length: int = 2048,
) -> np.ndarray:
    """Passes SMILES strings through PeptideCLM-2 and mean-pools valid token embeddings."""
    all_embeddings = []

    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i : i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            last_hidden = outputs.last_hidden_state  # (batch_size, seq_len, hidden_dim)

            # Mask out padding tokens.
            mask = inputs["attention_mask"].unsqueeze(-1)  # (batch_size, seq_len, 1)
            sum_embeddings = torch.sum(last_hidden * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)

            mean_pooled = (sum_embeddings / sum_mask).cpu().numpy()
            all_embeddings.append(mean_pooled)

    return np.vstack(all_embeddings)

## 4. Data cleaning, fidelity filtering, & deduplication

The Phase 1 dataset keeps one row per (peptide, assay-record) — duplicate rows sharing a `peptide_uid` are kept on purpose for masked multitask training (`data/README.md` §3.2), so we reduce to one row per unique peptide here for embedding.

This isn't a plain `drop_duplicates`, though. Checked directly against the real Phase 1 output: **39% of unique peptides (15,209 of 39,029) have more than one distinct SMILES string** across the rows sharing their `peptide_uid`. A 200-group sample showed ~58% of those are just alternate SMILES spellings of the same molecule (harmless — canonicalizing collapses them), but **~42% are genuinely different molecules** despite sharing a sequence+modification identity key (~16% of all unique peptides). Every row in an ambiguous group is already flagged `label_quality_flag == 'metadata_source_conflict'` by the dedup stage, which deliberately left the tie unresolved rather than picking a winner (see `dedup.py`'s `tied_disagree` branch).

So: canonicalize first (kills the harmless 58%), then break remaining ties with an explicit, deterministic rule — sorting by `source` matches the same alphabetical tie-break `dedup.py` itself already uses elsewhere — rather than relying on incidental row order. The conflict is carried forward as `has_smiles_conflict` so it's auditable against the cluster assignments later, instead of being silently resolved.

In [ ]:
raw_df = pd.read_parquet(DATASET_PATH)

# Step A: drop missing SMILES / failed-fidelity conversions.
df_clean = raw_df[raw_df["smiles"].notna() & (raw_df["smiles"] != "")].copy()
df_clean = df_clean[df_clean["chemical_fidelity_tier"] != "failed"]


# Step B: canonicalize SMILES so alternate-spelling duplicates of the same
# molecule collapse before the tie-break below has to deal with them.
def canonical_smiles(s):
    mol = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(mol) if mol is not None else s


df_clean["smiles_canonical"] = df_clean["smiles"].map(canonical_smiles)

# Step C: peptide_uid groups where sources disagree on the actual molecule
# (not just the SMILES string) -- flagged upstream by dedup.py.
conflicted_uids = set(
    df_clean.loc[df_clean["label_quality_flag"] == "metadata_source_conflict", "peptide_uid"]
)

# Step D: explicit, documented tie-break (matches dedup.py's own alphabetical-
# source convention) instead of relying on incidental row order.
df_clean = df_clean.sort_values("source")
df_dedup = df_clean.drop_duplicates(subset=["peptide_uid"], keep="first").reset_index(drop=True)
df_dedup["has_smiles_conflict"] = df_dedup["peptide_uid"].isin(conflicted_uids)

print(f"Data reduction: {len(raw_df)} raw rows -> {len(df_dedup)} unique peptide_uids for embedding.")
print(
    f"{df_dedup['has_smiles_conflict'].sum()} of those "
    f"({df_dedup['has_smiles_conflict'].mean():.1%}) carry an unresolved cross-source SMILES conflict."
)

## 5. Generate base embeddings

In [ ]:
unique_smiles = df_dedup["smiles_canonical"].tolist()

print(f"Extracting embeddings for {len(unique_smiles)} unique peptides...")
X_embeddings = extract_mean_pooled_embeddings(unique_smiles, batch_size=32, max_length=2048)

print(f"Embeddings matrix shape: {X_embeddings.shape}")

## 6. Joint K-Means clustering sweep (k=4–6)

Clustered on the union of all HC50 + MIC peptides together — per the roadmap, splitting the tasks independently here would leak across the shared encoder later.

In [ ]:
metrics_log = []

print("--- Joint K-Means sweep (k=4-6) ---")
for k in range(4, 7):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_embeddings)

    sil = silhouette_score(X_embeddings, labels)
    db = davies_bouldin_score(X_embeddings, labels)
    ch = calinski_harabasz_score(X_embeddings, labels)

    metrics_log.append({"k": k, "Silhouette": sil, "Davies_Bouldin": db, "Calinski_Harabasz": ch})
    print(f"k={k} | Silhouette: {sil:.4f} | DB: {db:.4f} | CH: {ch:.2f}")

results_df = pd.DataFrame(metrics_log)
best_k = int(results_df.loc[results_df["Silhouette"].idxmax()]["k"])
print(f"\nRecommended cluster count based on Silhouette score: k={best_k}")
print("Sanity-check against Davies-Bouldin (lower better) / Calinski-Harabasz (higher better) before locking this in --")
print("if they disagree with the Silhouette pick, don't accept it blindly.")

## 7. Attach cluster labels & export

In [ ]:
final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_dedup["cluster_label"] = final_kmeans.fit_predict(X_embeddings)

df_dedup[["peptide_uid", "smiles_canonical", "has_smiles_conflict", "cluster_label"]].to_csv(
    "phase2_joint_clusters.csv", index=False
)
np.save("phase2_embeddings.npy", X_embeddings)

print("Saved cluster mapping to 'phase2_joint_clusters.csv'")
print("Saved raw embeddings array to 'phase2_embeddings.npy'")